# Sliding Window


## Topic overview

Maintain a moving range that satisfies a constraint as it slides.

## Pattern-recognition rules

- Grow the window until invalid, then shrink from the left.
- Track a running aggregate (sum, count, min/max).

## Common data structures

- Arrays
- Strings

## Standard complexity expectations

- Amortized O(n) — each element enters and leaves at most once.

## Common mistakes

- Recomputing the aggregate from scratch each step (turns O(n) into O(nk)).

## Original illustrative example

In [ ]:
# Replace with an ORIGINAL example. Do not paste external
# problem statements. See src/algorithms/ for reusable helpers.
example_input = []
example_expected = None

## Add solved problems below

Each new sub-section should follow the template in `../templates/notebook_template.ipynb`.

1. [Substring with Concatenation of All Words](#substring-with-concatenation-of-all-words)


# Substring with Concatenation of All Words

## Metadata

- Source: NeetCode / LeetCode 30
- Problem URL: https://leetcode.com/problems/substring-with-concatenation-of-all-words/
- Difficulty: Hard
- Topic: Sliding Window
- Date started: 2026-09-14
- Date solved: 2026-09-14
- Current mastery level: 1
- Last reviewed: 2026-09-14
- Next review:


## Problem statement in my own words

You get a string `s` and a list `words`. Every entry of `words` has the
same length $w$.

A **concatenated substring** is a slice of `s` that is some permutation
of `words` glued end to end. Duplicates in `words` count: if `"word"`
appears twice, the slice must contain `"word"` twice.

```
words = ["ab", "cd", "ef"]
valid:   "abcdef", "abefcd", "cdabef", "cdefab", "efabcd", "efcdab"
invalid: "acdbef"   ← those letters appear, but not as those whole words
```

Return every start index in `s` where such a slice begins. Order of
indices does not matter.

You cannot afford to generate $m!$ permutations. The intended pass is a
word-aligned sliding window.


## Inputs, outputs, and constraints

- Inputs: `s` (string) and `words` (list of equal-length strings)
- Outputs: `list[int]` — starting indices of every concatenated slice
- Constraints:
  - $1 \le |s| \le 10^4$
  - $1 \le m \le 5000$ where $m = $ `len(words)`
  - $1 \le w \le 30$ where $w = $ `len(words[0])`
  - `s` and every word are lowercase English letters


## Examples

| Input | Expected | Notes |
|---|---|---|
| `s = "barfoothefoobarman"`, `words = ["foo", "bar"]` | `[0, 9]` | `"barfoo"` and `"foobar"` |
| `s = "wordgoodgoodgoodbestword"`, `words = ["word", "good", "best", "word"]` | `[]` | need *two* `"word"`; they never sit in one window with `"best"` |
| `s = "barfoofoobarthefoobarman"`, `words = ["bar", "foo", "the"]` | `[6, 9, 12]` | three overlapping 9-letter slices |

Window length is always $m \cdot w$. Example 1 uses $2 \times 3 = 6$.
Example 3 uses $3 \times 3 = 9$.


## Initial observations

- Characters are the wrong unit. The window must grow and shrink in
  steps of $w$, because a valid slice is a sequence of whole words.
- There are exactly $w$ alignments of `s`: start at index $0$, $1$,
  $\ldots$, $w-1$. Each alignment is an ordinary sliding-window
  problem on a stream of $w$-letter tokens.
- `Counter(words)` is the target bag. The live window is another
  `Counter`. A hit is `used == m` after both counters agree.
- An unknown token (not in `need`) poisons the window. Drop
  everything and jump `left` to `right`.
- Extra copies of a *needed* word are different: pop tokens from the
  left until that word's count is legal again. This is the usual
  "shrink while invalid" rule, one word at a time.
- After a hit, pop one word from the left and keep scanning. That is
  how overlapping answers like $6, 9, 12$ appear.
- Brute force tries every start and checks $m$ chunks. With
  $n = 10^4$, $m = 5000$, $w = 30$ that is around $10^9$ character
  reads. Too slow. The aligned window is $O(nw)$.


## Brute-force approach

### Why it works

At every index `i`, take the next $m$ word-sized chunks. If their
multiset equals `words`, record `i`. Correct, permutation-blind, and
too slow on the upper constraints.

### Implementation


In [ ]:
from collections import Counter
from typing import List


def brute_force(s: str, words: List[str]) -> List[int]:
    word_len = len(words[0])
    num_words = len(words)
    total = word_len * num_words
    need = Counter(words)
    result: List[int] = []

    for start in range(len(s) - total + 1):
        seen: Counter = Counter()
        valid = True
        for k in range(num_words):
            chunk_start = start + k * word_len
            word = s[chunk_start : chunk_start + word_len]
            if word not in need:
                valid = False
                break
            seen[word] += 1
            if seen[word] > need[word]:
                valid = False
                break
        if valid:
            result.append(start)
    return result


### Complexity

Let $n = |s|$, $m = $ `len(words)`, $w = $ `len(words[0])`.

- Time: $O(n m w)$ — up to $n$ starts, $m$ chunks, $w$ characters each
- Space: $O(m)$ for the two counters

Worst case $\approx 10^4 \cdot 5000 \cdot 30 = 1.5 \times 10^9$. Not viable.


## Optimized insight

Fix an alignment, then slide a window across *words*, not characters.

```
s:      b a r f o o t h e f o o b a r m a n
idx:    0 1 2 3 4 5 6 7 8 9 ...
offset 0 tokens:  bar | foo | the | foo | bar | man
offset 1 tokens:    arf | oot | hef | oob | arm | ...
offset 2 tokens:      rfo | oth | efo | oba | rma
```

Only offset $0$ can ever match `["foo", "bar"]`, because those words
are 3 letters and they sit on multiples of 3.

Each alignment is the classic shrink-while-invalid window:

1. eat one token on the right
2. if it is garbage, reset
3. if some needed word is over-count, spit tokens from the left
4. if the bag is exact, record `left`, then spit one token so the
   next overlap can be found

`used` is the number of tokens currently in the window. Because we
never let a count exceed `need`, `used == m` if and only if the
window is a permutation of `words`.

## Optimized approach

1. `need = Counter(words)`, `w = len(words[0])`, `m = len(words)`
2. For `offset` in $0 \ldots w-1$:
   - `left = right = offset`, empty `window`, `used = 0`
   - while `right + w <= n`:
     - take `word = s[right:right+w]`, advance `right` by $w$
     - if `word not in need`: clear, `left = right`, continue
     - else add it; while that word is over-count, drop from `left`
     - if `used == m`: append `left`, then drop one word from `left`
3. Return the collected indices

### Step-by-step trace — Example 1, offset 0

`s = "barfoothefoobarman"`, `words = ["foo", "bar"]`.
`need = {foo: 1, bar: 1}`, $w = 3$, $m = 2$.

| Step | left | word | right | window | used | Note |
|---|---|---|---|---|---|---|
| 1 | 0 | `bar` | 3 | `{bar:1}` | 1 | add |
| 2 | 0 | `foo` | 6 | `{bar:1, foo:1}` | 2 | **match 0**, then drop `bar` |
| 3 | 3 | `the` | 9 | `{}` | 0 | unknown → reset, `left = 9` |
| 4 | 9 | `foo` | 12 | `{foo:1}` | 1 | add |
| 5 | 9 | `bar` | 15 | `{foo:1, bar:1}` | 2 | **match 9**, then drop `foo` |
| 6 | 12 | `man` | 18 | `{}` | 0 | unknown → reset |

Offsets 1 and 2 never see `"foo"` or `"bar"` as tokens. Answer: `[0, 9]`.

### Trace — Example 3, offset 0

`s = "barfoofoobarthefoobarman"`, `words = ["bar", "foo", "the"]`.

Tokens: `bar foo foo bar the foo bar man`.

The second `"foo"` makes `window[foo] = 2 > 1`, so the window shrinks
until only one `"foo"` remains. After that the bag is `{foo, bar, the}`
three times in a row, starting at 6, then 9, then 12. Overlap comes
from dropping exactly one token after each hit.


In [ ]:
from collections import Counter
from typing import List


class Solution:
    def findSubstring(self, s: str, words: List[str]) -> List[int]:
        # Length of one word.
        word_len = len(words[0])

        # Number of words required.
        num_words = len(words)

        # Frequency table describing exactly what a valid window needs.
        need = Counter(words)

        result: List[int] = []

        # There are word_len possible alignments in the string.
        for offset in range(word_len):
            left = offset
            right = offset

            # Counts words currently inside the sliding window.
            window = Counter()

            # Number of complete words currently inside the window.
            used = 0

            while right + word_len <= len(s):
                # Read exactly one word-sized chunk.
                word = s[right:right + word_len]

                # Move right to the next word.
                right += word_len

                # --------------------------------------------------
                # Case 1: This word is not part of words at all.
                # --------------------------------------------------
                if word not in need:
                    window.clear()
                    used = 0
                    left = right
                    continue

                # --------------------------------------------------
                # Case 2: This is a required word.
                # --------------------------------------------------
                window[word] += 1
                used += 1

                # --------------------------------------------------
                # Case 3: We have too many copies of this word.
                #
                # Shrink the window from the left until its
                # frequency becomes valid again.
                # --------------------------------------------------
                while window[word] > need[word]:
                    left_word = s[left:left + word_len]

                    window[left_word] -= 1
                    used -= 1

                    left += word_len

                # --------------------------------------------------
                # Case 4: The window contains exactly all required
                # words.
                # --------------------------------------------------
                if used == num_words:
                    result.append(left)

                    # Remove one word from the left so that we can
                    # continue searching for overlapping matches.
                    left_word = s[left:left + word_len]

                    window[left_word] -= 1
                    used -= 1

                    left += word_len

        return result


def optimized(s: str, words: List[str]) -> List[int]:
    return Solution().findSubstring(s, words)


## Complexity analysis

There are $w$ alignments. Across one alignment each token of that
alignment enters the window once and leaves at most once, so the
inner work is $O(n / w)$ token steps. Each token is a slice of $w$
characters.

$$
w \text{ offsets} \times O(n / w) \text{ tokens} \times O(w) \text{ to read} = O(n w)
$$

Counter updates are $O(1)$ average.

- Time: $O(n w)$
- Space: $O(m)$ for `need` and `window`

With the given limits that is about $10^4 \times 30 = 3 \times 10^5$
character reads.


## Edge cases

- $m = 1$: reduces to "find all occurrences of this one word"
- $n < m w$: no window fits; return `[]`
- duplicate words (`["word", "word"]`): `need` must keep the count
- extra copies of a needed word in `s`: shrink, do not reset
- unknown token in the middle: full reset, `left = right`
- overlapping hits (example 3): must pop one word after a match
- several alignments all producing hits (`"aaaaaaaa"` / `["aa", "aa"]`)
- answer order is free; do not sort unless a test asks for it


## Testing


In [ ]:
assert optimized("barfoothefoobarman", ["foo", "bar"]) == [0, 9]
assert optimized("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]) == []
assert optimized("barfoofoobarthefoobarman", ["bar", "foo", "the"]) == [6, 9, 12]
assert optimized("a", ["a"]) == [0]
assert sorted(optimized("aaaaaaaa", ["aa", "aa"])) == [0, 1, 2, 3, 4]
assert brute_force("barfoothefoobarman", ["foo", "bar"]) == [0, 9]
assert brute_force("wordgoodgoodgoodbestword", ["word", "good", "best", "word"]) == []
assert brute_force("barfoofoobarthefoobarman", ["bar", "foo", "the"]) == [6, 9, 12]
assert brute_force("a", ["a"]) == [0]

print("Substring with Concatenation of All Words checks passed")


## Visualization

Two views of the same idea:

1. **Alignments** — $w$ ways to tile `s` with word-sized slots. A
   valid concat can only start on one of those grids.
2. **Window walk** — each frame is one token step of `findSubstring`.
   Blue is the live window, orange is the token just read, green is a
   reported match. The right panel is `need` vs `window`.

If `ipywidgets` is installed, use **Next step** / **Prev** / **Reset**.
Otherwise every iteration is drawn as a static row of frames.


In [ ]:
"""Standalone visualization for Substring with Concatenation of All Words."""

from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, field
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


@dataclass
class ConcatFrame:
    step: int
    offset: int
    left: int
    right: int
    word: str
    word_start: int
    window: Dict[str, int]
    used: int
    result: List[int]
    decision: str
    match_at: Optional[int] = None
    need: Dict[str, int] = field(default_factory=dict)


def concat_frames(s: str, words: List[str], *, only_offset: Optional[int] = None) -> List[ConcatFrame]:
    """Record window state using the same control flow as findSubstring."""
    word_len = len(words[0])
    num_words = len(words)
    need = Counter(words)
    frames: List[ConcatFrame] = []
    result: List[int] = []
    step = 1
    offsets = [only_offset] if only_offset is not None else list(range(word_len))

    def snap(
        offset: int,
        left: int,
        right: int,
        word: str,
        word_start: int,
        window: Counter,
        used: int,
        decision: str,
        match_at: Optional[int] = None,
    ) -> None:
        nonlocal step
        frames.append(
            ConcatFrame(
                step,
                offset,
                left,
                right,
                word,
                word_start,
                {k: v for k, v in window.items() if v},
                used,
                list(result),
                decision,
                match_at=match_at,
                need=dict(need),
            )
        )
        step += 1

    for offset in offsets:
        left = offset
        right = offset
        window: Counter = Counter()
        used = 0

        while right + word_len <= len(s):
            word_start = right
            word = s[right : right + word_len]
            right += word_len

            if word not in need:
                snap(
                    offset,
                    left,
                    right,
                    word,
                    word_start,
                    window,
                    used,
                    f"'{word}' not in words  →  reset window",
                )
                window.clear()
                used = 0
                left = right
                continue

            window[word] += 1
            used += 1

            while window[word] > need[word]:
                left_word = s[left : left + word_len]
                window[left_word] -= 1
                used -= 1
                left += word_len
                snap(
                    offset,
                    left,
                    right,
                    word,
                    word_start,
                    window,
                    used,
                    f"too many '{word}'  →  drop '{left_word}' from left",
                )

            if used == num_words:
                result.append(left)
                snap(
                    offset,
                    left,
                    right,
                    word,
                    word_start,
                    window,
                    used,
                    f"MATCH at index {left}  ('{s[left:right]}')",
                    match_at=left,
                )
                left_word = s[left : left + word_len]
                window[left_word] -= 1
                used -= 1
                left += word_len
                continue

            snap(
                offset,
                left,
                right,
                word,
                word_start,
                window,
                used,
                f"added '{word}'   used={used}/{num_words}",
            )

    return frames


def plot_alignments(s: str, word_len: int) -> None:
    """Show why we launch one scan per residue mod word_len."""
    fig, axes = plt.subplots(word_len, 1, figsize=(max(10, len(s) * 0.55), 1.35 * word_len), sharex=True)
    if word_len == 1:
        axes = [axes]
    for offset, ax in enumerate(axes):
        ax.set_xlim(-0.5, len(s) - 0.5)
        ax.set_ylim(-0.6, 1.4)
        ax.axis("off")
        ax.set_title(f"offset {offset}  —  read s[{offset}::{word_len}]", loc="left", fontsize=10)
        for i, ch in enumerate(s):
            grouped = (i - offset) // word_len if i >= offset else -1
            color = "#4c78a8" if i >= offset else "#d9d9d9"
            if grouped >= 0 and grouped % 2 == 1:
                color = "#73a3c9" if i >= offset else "#d9d9d9"
            ax.add_patch(
                Rectangle((i - 0.45, 0), 0.9, 1, facecolor=color, edgecolor="#1d3557", linewidth=0.6)
            )
            ax.text(i, 0.5, ch, ha="center", va="center", fontsize=11, color="white", fontweight="bold")
            ax.text(i, -0.28, str(i), ha="center", va="top", fontsize=7, color="#6b6b6b")
        for k in range(offset, len(s) - word_len + 1, word_len):
            ax.plot([k - 0.45, k + word_len - 0.55], [-0.48, -0.48], color="#9b2226", linewidth=2)
    fig.suptitle(f's = "{s}"   word length = {word_len}', fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_concat_frame(s: str, words: List[str], frame: ConcatFrame, *, ax_str=None, ax_cnt=None):
    word_len = len(words[0])
    num_words = len(words)
    created = ax_str is None
    if created:
        fig, (ax_str, ax_cnt) = plt.subplots(
            1, 2, figsize=(max(12, len(s) * 0.62 + 4), 4.2), gridspec_kw={"width_ratios": [2.2, 1]}
        )

    ax_str.set_xlim(-0.6, len(s) - 0.4)
    ax_str.set_ylim(-1.15, 1.7)
    ax_str.axis("off")

    for i, ch in enumerate(s):
        if frame.match_at is not None and frame.match_at <= i < frame.match_at + word_len * num_words:
            facecolor = "#2a9d8f"
        elif frame.word_start <= i < frame.word_start + word_len:
            facecolor = "#e76f51"
        elif frame.left <= i < frame.right:
            facecolor = "#4c78a8"
        else:
            facecolor = "#d9d9d9"
        ax_str.add_patch(
            Rectangle((i - 0.45, 0), 0.9, 1, facecolor=facecolor, edgecolor="#1d3557", linewidth=0.6)
        )
        ax_str.text(i, 0.5, ch, ha="center", va="center", fontsize=11, color="white", fontweight="bold")
        ax_str.text(i, -0.22, str(i), ha="center", va="top", fontsize=7, color="#6b6b6b")

    ax_str.annotate(
        "L",
        xy=(frame.left, 1.05),
        ha="center",
        fontsize=10,
        fontweight="bold",
        color="#1d3557",
    )
    ax_str.annotate(
        "R",
        xy=(max(frame.right - 0.5, frame.left), 1.05),
        ha="center",
        fontsize=10,
        fontweight="bold",
        color="#1d3557",
    )
    ax_str.set_title(
        f"Step {frame.step}  offset={frame.offset}  L={frame.left}  R={frame.right}"
        f"  used={frame.used}/{num_words}  found={frame.result}\n{frame.decision}",
        loc="left",
        fontsize=10,
    )

    labels = sorted(frame.need.keys())
    xs = list(range(len(labels)))
    need_vals = [frame.need[w] for w in labels]
    have_vals = [frame.window.get(w, 0) for w in labels]
    ax_cnt.bar([x - 0.18 for x in xs], need_vals, width=0.36, color="#9a9a9a", label="need")
    colors = ["#2a9d8f" if have_vals[i] == need_vals[i] else "#e76f51" if have_vals[i] > need_vals[i] else "#4c78a8" for i in xs]
    ax_cnt.bar([x + 0.18 for x in xs], have_vals, width=0.36, color=colors, label="window")
    ax_cnt.set_xticks(xs)
    ax_cnt.set_xticklabels(labels)
    ax_cnt.set_ylabel("count")
    ax_cnt.set_ylim(0, max(need_vals + have_vals + [1]) + 0.8)
    ax_cnt.set_title("need vs window")
    ax_cnt.legend(fontsize=8, loc="upper right")

    if created:
        plt.tight_layout()
    return ax_str, ax_cnt


def visualize_concat_all(s: str, words: List[str], *, only_offset: Optional[int] = None) -> None:
    frames = concat_frames(s, words, only_offset=only_offset)
    print(f's="{s}"  words={words}')
    print(f"{'step':>4}  {'off':>3}  {'L':>3}  {'R':>3}  word    decision")
    for frame in frames:
        print(
            f"{frame.step:>4}  {frame.offset:>3}  {frame.left:>3}  {frame.right:>3}  "
            f"{frame.word:<6}  {frame.decision}"
        )
        plot_concat_frame(s, words, frame)
    plt.show()


def step_through_concat(s: str, words: List[str], *, only_offset: Optional[int] = None) -> None:
    frames = concat_frames(s, words, only_offset=only_offset)
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output, display
    except ImportError:
        visualize_concat_all(s, words, only_offset=only_offset)
        return

    output = widgets.Output()
    state = {"i": 0}

    def render() -> None:
        with output:
            clear_output(wait=True)
            frame = frames[state["i"]]
            print(f'Example s="{s}"  words={words}   frame {state["i"] + 1}/{len(frames)}')
            plot_concat_frame(s, words, frame)
            plt.show()

    def on_next(_=None) -> None:
        state["i"] = min(state["i"] + 1, len(frames) - 1)
        render()

    def on_prev(_=None) -> None:
        state["i"] = max(state["i"] - 1, 0)
        render()

    def on_reset(_=None) -> None:
        state["i"] = 0
        render()

    prev_btn = widgets.Button(description="Prev")
    next_btn = widgets.Button(description="Next step")
    reset_btn = widgets.Button(description="Reset")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    reset_btn.on_click(on_reset)
    display(widgets.HBox([prev_btn, next_btn, reset_btn]), output)
    render()

print("Word-aligned scans of example 1")
plot_alignments("barfoothefoobarman", 3)

print()
print("Example 1 — walk offset 0 (the productive alignment)")
step_through_concat("barfoothefoobarman", ["foo", "bar"], only_offset=0)

print()
print("Example 3 — overlapping matches on offset 0")
step_through_concat("barfoofoobarthefoobarman", ["bar", "foo", "the"], only_offset=0)


## Alternative approaches

- Generate every permutation of `words` and `str.find` each: $O(m!)$
  strings, unusable for $m > 8$.
- Hash each word and compare rolling hashes of $m$-token windows:
  same $O(nw)$ idea, fussier constants.
- Recurse / backtrack on the next unused word at every index: correct
  exponential search, not the intended solution.
- Character-level anagram window of length $mw$: too loose. `"acdbef"`
  is an anagram of `"abcdef"` but not a word-permutation.

## Mistakes I made

- Sliding by 1 without the $w$ alignments. Still correct if you reset
  carefully, but slower to reason about and easy to desync `left`.
- Resetting on extra copies of a *needed* word. Extra `"foo"` must
  shrink, not wipe the window — that is how example 3 works.
- Forgetting to pop one word after a match, so overlapping starts
  like $6, 9, 12$ disappear.
- Using `==` on two `Counter`s every step instead of `used == m`.
  Both work; `used` is enough because counts are clamped to `need`.
- Generating permutations. If you reach `itertools.permutations`,
  stop and rebuild the window.

## Pattern recognition

Bag-of-tokens window, not bag-of-characters:

```python
for offset in range(word_len):
    # ordinary sliding window on s[offset::] in steps of word_len
    # unknown token → reset
    # over-count → shrink from left
    # used == m → record left, then shrink by one
```

Same muscle as "Find All Anagrams in a String", with the alphabet
replaced by equal-length words.

## Related problems

- Find All Anagrams in a String — this problem with $w = 1$
- Permutation in String — boolean version of anagrams
- Minimum Window Substring — shrink to the *smallest* covering window
- Sliding Window Maximum — deque instead of a `Counter`
- Repeated DNA Sequences — fixed-length chunks, different check

## Real-world or engineering connection

This is motif search over a token stream: DNA $k$-mers, protocol
frames of fixed size, or n-gram bags in a text pipeline. You do not
rebuild every permutation of the motif; you keep a running bag and
slide it across alignments of the record.

## Final takeaways

Do not permute `words`. Keep this:

1. $w$ alignments, windows that move $w$ characters at a time
2. `need` vs `window` plus an `used` counter
3. unknown token resets; extra needed token shrinks
4. after a hit, drop one word and keep going

The reusable pattern is a sliding bag of equal-length tokens.


## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-09-14 | 1 | First write-up: word-aligned window, reset vs shrink, overlap by popping one |
